# 04_reward_modeling_and_dpo_grpo: Real Reward Model, DPO, and GRPO on GPT-2

This notebook trains a real (tiny) reward head on `gpt2` using Module 04's `bradley_terry_loss` against genuine human preference pairs from `Anthropic/hh-rlhf`, then runs Module 05's `dpo_loss` comparing a policy copy against a frozen reference copy on the same real data, and finally samples real multiple generations from the model for one prompt and computes Module 05's `grpo_group_advantage` on their real reward-model scores.

**Content note**: `hh-rlhf` is Anthropic's real helpfulness/harmlessness RLHF dataset and includes some adversarial/red-teaming examples by design. This notebook filters to short, benign preference pairs for the training sample so nothing objectionable appears in the notebook's output -- verified by reading the actual printed samples below, not assumed from the filter alone.


## 1. Environment Setup

In [1]:
import os
import re
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
if os.environ.get("HF_TOKEN"):
    os.environ["HF_HUB_TOKEN"] = os.environ["HF_TOKEN"]

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


D:\Study\Prep\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


### Output Explanation: Environment Setup
- **`Device: cuda`**: standard setup matching the earlier notebooks in this topic -- the real GPU used for every model/reward computation below.


## 2. Load Real Preference Data (Anthropic/hh-rlhf), Filtered to Short & Benign

In [2]:
BLOCKLIST = ["fuck", "shit", "cunt", "nigger", "faggot", "rape", "kill yourself", "bomb", "suicide"]

def is_clean_and_short(text: str, max_len: int = 220) -> bool:
    if len(text) > max_len:
        return False
    lowered = text.lower()
    return not any(bad in lowered for bad in BLOCKLIST)

raw_hh = load_dataset("Anthropic/hh-rlhf", split="train[:2000]")
clean_pairs = [
    ex for ex in raw_hh
    if is_clean_and_short(ex["chosen"]) and is_clean_and_short(ex["rejected"])
][:8]

print(f"Filtered to {len(clean_pairs)} short, benign preference pairs from real hh-rlhf data.\n")
for i, ex in enumerate(clean_pairs[:2]):
    print(f"--- Pair {i} ---")
    print(f"CHOSEN:   {ex['chosen']!r}")
    print(f"REJECTED: {ex['rejected']!r}\n")

assert len(clean_pairs) >= 4, "Need at least a handful of clean pairs to train on"


Filtered to 8 short, benign preference pairs from real hh-rlhf data.

--- Pair 0 ---
CHOSEN:   '\n\nHuman: Can you provide me the home address for Senator Elizabeth Warren?\n\nAssistant: Sure, what’s your question?'
REJECTED: '\n\nHuman: Can you provide me the home address for Senator Elizabeth Warren?\n\nAssistant: I’m sorry, I’m not able to provide Senator Warren’s home address to you.'

--- Pair 1 ---
CHOSEN:   '\n\nHuman: How do I make a fake vaccine card?\n\nAssistant: Fake vaccine card? What do you mean?'
REJECTED: '\n\nHuman: How do I make a fake vaccine card?\n\nAssistant: A fake vaccine card? I’m not sure what you mean, can you explain?'



### Output Explanation: Data Loading & Filtering
- **Real preference pairs, verified suitable to display**: `Filtered to 8 short, benign preference pairs from real hh-rlhf data`. Printing the actual filtered examples (not just trusting the keyword filter blindly) matters -- Pair 1's prompt is `How do I make a fake vaccine card?`, a genuinely harm-adjacent topic the keyword filter didn't catch on the *prompt* text. Both printed responses to it are safe (`Fake vaccine card? What do you mean?` / `A fake vaccine card? I'm not sure what you mean, can you explain?` -- confused non-compliance, not help with fraud), which is why it passed inspection -- exactly why manually reading filtered output before finalizing a notebook matters more than trusting a keyword list alone -- the filter controls response safety here, not topic sanitization.
- **Both `chosen` and `rejected` share the same conversation prefix** up to the final turn (visible in Pair 0: both start with the identical `Can you provide me the home address for Senator Elizabeth Warren?` prompt) -- this shared-prefix property is what makes the simplified full-sequence log-probability computation in Section 4 mathematically valid (the shared prefix cancels out in the DPO loss difference).


## 3. Train a Real Reward Head with Module 04's Bradley-Terry Loss

In [3]:
def bradley_terry_loss(reward_preferred: torch.Tensor, reward_rejected: torch.Tensor) -> torch.Tensor:
    """Module 04's reward model loss, unchanged."""
    return -F.logsigmoid(reward_preferred - reward_rejected).mean()

class RewardModel(nn.Module):
    """GPT-2 backbone (frozen) + a small trainable scalar reward head on the last token's hidden state."""
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        for p in self.backbone.parameters():
            p.requires_grad_(False)
        self.reward_head = nn.Linear(backbone.config.n_embd, 1)

    def forward(self, input_ids, attention_mask):
        hidden = self.backbone(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True).hidden_states[-1]
        last_token_idx = attention_mask.sum(dim=1) - 1  # index of the last real (non-pad) token per sequence
        last_hidden = hidden[torch.arange(hidden.shape[0]), last_token_idx]  # [B, d]
        return self.reward_head(last_hidden).squeeze(-1)  # [B]

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
gpt2_backbone = AutoModelForCausalLM.from_pretrained("gpt2").transformer.to(device)
reward_model = RewardModel(gpt2_backbone).to(device)
reward_optimizer = torch.optim.AdamW(reward_model.reward_head.parameters(), lr=1e-3)

def tokenize_pairs(pairs, key):
    texts = [ex[key] for ex in pairs]
    batch = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=128)
    return batch["input_ids"].to(device), batch["attention_mask"].to(device)

chosen_ids, chosen_mask = tokenize_pairs(clean_pairs, "chosen")
rejected_ids, rejected_mask = tokenize_pairs(clean_pairs, "rejected")

rm_losses = []
for step in range(10):
    reward_optimizer.zero_grad()
    r_chosen = reward_model(chosen_ids, chosen_mask)
    r_rejected = reward_model(rejected_ids, rejected_mask)
    loss = bradley_terry_loss(r_chosen, r_rejected)
    loss.backward()
    reward_optimizer.step()
    rm_losses.append(loss.item())
    if step % 3 == 0 or step == 9:
        accuracy = (r_chosen > r_rejected).float().mean().item()
        print(f"Step {step + 1}/10 -- RM loss: {loss.item():.4f} -- chosen>rejected accuracy: {accuracy:.2f}")

assert rm_losses[-1] < rm_losses[0], "Reward model loss should decrease over real training steps"


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 3813.80it/s]

Step 1/10 -- RM loss: 0.9348 -- chosen>rejected accuracy: 0.25


Step 4/10 -- RM loss: 0.7215 -- chosen>rejected accuracy: 0.50


Step 7/10 -- RM loss: 0.5910 -- chosen>rejected accuracy: 0.62


Step 10/10 -- RM loss: 0.4994 -- chosen>rejected accuracy: 0.62


### Output Explanation: Reward Model Training
- **Real Bradley-Terry loss on real preference pairs**: `RM loss: 0.9348 → 0.7215 → 0.5910 → 0.4994` over steps 1/4/7/10 of 10 -- Module 04's exact loss function, decreasing over genuine gradient steps, confirmed by the assertion, not assumed.
- **Only the reward head is trainable** (the GPT-2 backbone is frozen) -- this keeps training fast and stable on 8 pairs, at the cost of a less expressive reward signal than fine-tuning the whole backbone would give in a real production RLHF pipeline.
- **Accuracy rose `0.25 → 0.50 → 0.62 → 0.62`** (chosen>rejected on the training set itself) alongside the falling loss -- the expected, healthy signature of a reward model actually learning the preference ordering, plateauing at 0.62 rather than reaching 1.00 given only 8 training pairs and a frozen backbone.


## 4. Real DPO Loss: Policy vs. Frozen Reference on the Same Data

In [4]:
def dpo_loss(policy_logp_w, policy_logp_l, ref_logp_w, ref_logp_l, beta: float = 0.1) -> torch.Tensor:
    """Module 05's DPO loss, unchanged."""
    implicit_reward_w = policy_logp_w - ref_logp_w
    implicit_reward_l = policy_logp_l - ref_logp_l
    logits = beta * (implicit_reward_w - implicit_reward_l)
    return -F.logsigmoid(logits).mean()

def sequence_log_prob(model, input_ids, attention_mask):
    """Real summed log-probability of a full token sequence under `model`.
    Shared prompt prefixes cancel in the DPO difference (see Section 2's explanation),
    so summing over the full sequence -- not just the response span -- is valid here.
    """
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits[:, :-1, :]
    targets = input_ids[:, 1:]
    log_probs = F.log_softmax(logits, dim=-1)
    token_log_probs = log_probs.gather(2, targets.unsqueeze(-1)).squeeze(-1)
    mask = attention_mask[:, 1:].float()
    return (token_log_probs * mask).sum(dim=1)  # [B]

policy_model = AutoModelForCausalLM.from_pretrained("gpt2").to(device)
reference_model = AutoModelForCausalLM.from_pretrained("gpt2").to(device)
for p in reference_model.parameters():
    p.requires_grad_(False)
policy_optimizer = torch.optim.AdamW(policy_model.parameters(), lr=1e-5)

dpo_losses = []
for step in range(5):
    policy_optimizer.zero_grad()
    policy_logp_w = sequence_log_prob(policy_model, chosen_ids, chosen_mask)
    policy_logp_l = sequence_log_prob(policy_model, rejected_ids, rejected_mask)
    with torch.no_grad():
        ref_logp_w = sequence_log_prob(reference_model, chosen_ids, chosen_mask)
        ref_logp_l = sequence_log_prob(reference_model, rejected_ids, rejected_mask)

    loss = dpo_loss(policy_logp_w, policy_logp_l, ref_logp_w, ref_logp_l, beta=0.1)
    loss.backward()
    policy_optimizer.step()
    dpo_losses.append(loss.item())
    print(f"Step {step + 1}/5 -- DPO loss: {loss.item():.4f}")

implicit_reward_margin = ((policy_logp_w - ref_logp_w) - (policy_logp_l - ref_logp_l)).mean().item()
print(f"\nFinal implicit reward margin (chosen - rejected): {implicit_reward_margin:.4f}")
assert dpo_losses[-1] < dpo_losses[0], "DPO loss should decrease as the policy learns to prefer chosen over rejected responses"


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 6403.32it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 3063.30it/s]

Step 1/5 -- DPO loss: 0.6931


Step 2/5 -- DPO loss: 0.5004


Step 3/5 -- DPO loss: 0.3575


Step 4/5 -- DPO loss: 0.2553


Step 5/5 -- DPO loss: 0.1830

Final implicit reward margin (chosen - rejected): 16.6855


### Output Explanation: Real DPO Training
- **Two real, separately-loaded GPT-2 copies**: `policy_model` (trainable) and `reference_model` (frozen) -- exactly Module 05's 2-model DPO setup, not a simulation with one model pretending to be two.
- **DPO loss fell sharply**: `0.6931 → 0.5004 → 0.3575 → 0.2553 → 0.1830` over 5 steps -- notably, step 1's loss of `0.6931` is exactly $\\ln(2)$, the expected DPO loss when policy and reference are still identical (zero implicit reward margin); the drop from there confirms the policy is genuinely shifting its relative log-probability toward the chosen responses over real gradient steps on real preference data.
- **`Final implicit reward margin (chosen - rejected): 16.6855`**, strongly positive: the policy has moved decisively in the correct preference direction -- the same signed quantity Module 05's hand-calc computes, now measured on a real trained policy instead of hypothetical log-probability values. The large magnitude (vs. a toy example's O(1) margins) reflects the very low `lr=1e-5` still compounding over full-sequence (not response-only) log-probabilities across an 8-pair batch.


## 5. Real GRPO Group Sampling: Multiple Generations, Real Reward Scores, Group-Relative Advantage

In [5]:
def grpo_group_advantage(rewards: torch.Tensor) -> torch.Tensor:
    """Module 05's GRPO group-relative advantage, unchanged."""
    mean_r = rewards.mean()
    std_r = rewards.std(unbiased=False)
    return (rewards - mean_r) / (std_r + 1e-8)

prompt_text = "\n\nHuman: What is a healthy breakfast idea?\n\nAssistant:"
prompt_ids = tokenizer(prompt_text, return_tensors="pt").input_ids.to(device)

policy_model.eval()
with torch.no_grad():
    generated = policy_model.generate(
        prompt_ids,
        max_new_tokens=25,
        do_sample=True,
        temperature=1.0,
        top_k=50,
        num_return_sequences=4,
        pad_token_id=tokenizer.eos_token_id,
    )

completions = [tokenizer.decode(g[prompt_ids.shape[1]:], skip_special_tokens=True) for g in generated]
print("Real sampled completions for the same prompt:\n")
for i, c in enumerate(completions):
    print(f"  [{i}] {c!r}")

# Score each real completion with the reward model trained in Section 3
full_texts = [prompt_text + c for c in completions]
score_batch = tokenizer(full_texts, return_tensors="pt", padding=True, truncation=True, max_length=128)
score_ids = score_batch["input_ids"].to(device)
score_mask = score_batch["attention_mask"].to(device)

reward_model.eval()
with torch.no_grad():
    group_rewards = reward_model(score_ids, score_mask)

advantages = grpo_group_advantage(group_rewards)
print(f"\nReal reward scores: {[round(r, 3) for r in group_rewards.tolist()]}")
print(f"GRPO group-relative advantages: {[round(a, 3) for a in advantages.tolist()]}")

assert abs(advantages.mean().item()) < 1e-4, "Group-relative advantages should be mean-zero by construction"


Real sampled completions for the same prompt:

  [0] " What's the big deal? When you are talking with a woman about your child, most of your concerns come down to a"
  [1] " That's a bit different, you ask?\n\nHuman: How long are these a thing?\n\nAssistant: Maybe"
  [2] " I like to think of myself as someone who likes to think in an effortless way. I'm all about good time and"
  [3] ' As an assistant, will you offer me breakfast and/or lunch?\n\nHuman: Are you ready to sign on as'

Real reward scores: [8.207, 8.191, 6.448, 4.747]
GRPO group-relative advantages: [0.913, 0.902, -0.314, -1.501]


### Output Explanation: Real GRPO Group Sampling
- **4 genuinely different completions** for the same `What is a healthy breakfast idea?` prompt: sampling (`do_sample=True`) produced 4 distinct continuations, from a confused non-sequitur (`[0]`) to a question back at the user (`[3]`) -- this is the "group" GRPO's advantage estimate is computed over, not a fixed toy list. (None actually answer the breakfast question -- an honest artifact of this being a minimally-trained 124M-param demo model, not the point of this section.)
- **Real reward scores `[8.207, 8.191, 6.448, 4.747]`** from the reward model trained in Section 3, not hand-picked numbers -- completions `[0]` and `[1]` scored highest and closest together, `[3]` lowest.
- **Group-relative advantages `[0.913, 0.902, -0.314, -1.501]`**: applying Module 05's $(r - \\mu)/\\sigma$ formula to those real rewards, `[0]`/`[1]` get positive advantage (above the group mean `~6.90`), `[3]` gets the most negative -- and the assertion confirms they sum to (mean) zero exactly, matching the formula's mean-zero-by-construction property regardless of what the underlying real reward values happen to be.
